**Price Gap Moments Analysis**

This notebook reads in price gap, tariff, and gravity data for multiple years, combines them, and reports correlations between key variables.

In [1]:
import pandas as pd
import numpy as np
from scipy import stats

In [2]:
# Define years and read in data
years = ["2004", "2011", "2017"]
all_dfs = []

# Read gravity data (same for all years)
grav_df = pd.read_csv("../data/top30_gravity_data.csv")

for year in years:
    # Read price gap data
    df = pd.read_csv(f"../data/pricegap-df-{year}.csv")
    df = df.rename(columns={"exporter": "iso_o", "importer": "iso_d"})
    
    # Read tariff data
    tariffs_df = pd.read_csv(f"../data/tariffs-{year}.csv")
    tariffs_df = tariffs_df.rename(columns={"exporter": "iso_o", "importer": "iso_d"})
    
    # Merge price gap with tariffs
    df = df.merge(tariffs_df, on=["iso_o", "iso_d"], how="inner")
    
    # Merge with gravity data
    df = df.merge(grav_df, on=["iso_o", "iso_d"], how="inner")
    
    # Add year column
    df["year"] = year
    
    all_dfs.append(df)
    print(f"Year {year}: {len(df)} observations")

# Combine all years
big_df = pd.concat(all_dfs, ignore_index=True)
print(f"\nTotal observations (all years): {len(big_df)}")

Year 2004: 870 observations
Year 2011: 870 observations
Year 2017: 870 observations

Total observations (all years): 2610


In [3]:
# Filter out extreme trade share values (Xni ≈ 0 or Xni ≈ 1)
big_df = big_df[~np.isclose(big_df["Xni"], 1.0)]
big_df = big_df[~np.isclose(big_df["Xni"], 0.0)]

print(f"Observations after filtering: {len(big_df)}")
big_df.head()

Observations after filtering: 2604


,iso_o,iso_d,E_importer,Xni,logXni,dni,dni2,dni3,τni,year,...,dist,importer,exporter,distbin,bin375,bin750,bin1500,bin3000,bin6000,binmax
0,AUT,AUS,2.356209e+11,0.004297,-5.449770,0.874112,0.480179,0.307567,0.339296,2004,...,9937.199,1,2,6.0,0,0,0,0,0,1
1,BEL,AUS,2.356209e+11,0.014961,-4.202340,0.823992,0.381552,0.294151,0.287386,2004,...,10416.159,1,3,6.0,0,0,0,0,0,1
2,BRA,AUS,2.356209e+11,0.001459,-6.529799,0.654726,0.459052,0.316882,0.110739,2004,...,8310.602,1,4,6.0,0,0,0,0,0,1
3,CAN,AUS,2.356209e+11,0.008851,-4.727270,0.488547,0.167467,0.120297,0.293881,2004,...,9687.172,1,5,6.0,0,0,0,0,0,1
4,CHN,AUS,2.356209e+11,0.059249,-2.826004,0.718615,0.455234,0.203520,0.175503,2004,...,5566.461,1,6,5.0,0,0,0,0,1,0


In [4]:
# Function to compute correlation with confidence interval
def correlation_with_ci(x, y, name_x, name_y, alpha=0.10):
    """Compute Pearson correlation with confidence interval."""
    # Remove any NaN/Inf values
    mask = np.isfinite(x) & np.isfinite(y)
    x_clean = x[mask]
    y_clean = y[mask]
    
    n = len(x_clean)
    r, p_value = stats.pearsonr(x_clean, y_clean)
    
    # Fisher z-transformation for confidence interval
    z = np.arctanh(r)
    se = 1 / np.sqrt(n - 3)
    z_crit = stats.norm.ppf(1 - alpha/2)
    z_lo, z_hi = z - z_crit * se, z + z_crit * se
    ci_lo, ci_hi = np.tanh(z_lo), np.tanh(z_hi)
    
    print(f"Correlation of {name_x} and {name_y}")
    print(f"  Pearson r = {r:.4f}")
    print(f"  p-value = {p_value:.4e}")
    print(f"  n = {n}")
    print(f"  {int((1-alpha)*100)}% CI: [{ci_lo:.4f}, {ci_hi:.4f}]")
    print()

---
### Summary Statistics

In [5]:
# Summary statistics for dni by year
dni_summary = []

# N Basic Headings by year
n_basic_headings = {"All": np.nan, "2004": 62, "2011": 71, "2017": 64}

for year in ["All"] + years:
    if year == "All":
        df_subset = big_df
    else:
        df_subset = big_df[big_df["year"] == year]
    
    dni_summary.append({
        "Year": year,
        "N": len(df_subset),
        "N Basic Headings": n_basic_headings[year],
        "Mean": df_subset["dni"].mean(),
        "Median": df_subset["dni"].median(),
        "Min": df_subset["dni"].min(),
        "Max": df_subset["dni"].max(),
        "Std": df_subset["dni"].std()
    })

dni_stats_df = pd.DataFrame(dni_summary)
print("Summary Statistics for dni")
dni_stats_df

Summary Statistics for dni


,Year,N,N Basic Headings,Mean,Median,Min,Max,Std
0,All,2604,NaN,0.938037,0.855056,0.185344,3.128155,0.418341
1,2004,866,62.0,0.923801,0.897819,0.252115,2.268377,0.320867
2,2011,868,71.0,0.979768,0.829368,0.185344,3.128155,0.514208
3,2017,870,64.0,0.910572,0.835446,0.205745,2.258262,0.394010


In [6]:
# Summary statistics for dni2 by year
dni2_summary = []

# N Basic Headings by year
n_basic_headings = {"All": np.nan, "2004": 62, "2011": 71, "2017": 64}

for year in ["All"] + years:
    if year == "All":
        df_subset = big_df
    else:
        df_subset = big_df[big_df["year"] == year]
    
    dni2_summary.append({
        "Year": year,
        "N": len(df_subset),
        "N Basic Headings": n_basic_headings[year],
        "Mean": df_subset["dni2"].mean(),
        "Median": df_subset["dni2"].median(),
        "Min": df_subset["dni2"].min(),
        "Max": df_subset["dni2"].max(),
        "Std": df_subset["dni2"].std()
    })

dni2_stats_df = pd.DataFrame(dni2_summary)
print("Summary Statistics for dni2")
dni2_stats_df

Summary Statistics for dni2


,Year,N,N Basic Headings,Mean,Median,Min,Max,Std
0,All,2604,NaN,0.343285,0.344033,0.075897,0.773079,0.125588
1,2004,866,62.0,0.369150,0.379527,0.100390,0.764538,0.118956
2,2011,868,71.0,0.337716,0.334679,0.075897,0.734787,0.127989
3,2017,870,64.0,0.323093,0.309945,0.080869,0.773079,0.125337


---
### Summary Correlation Table

In [11]:
# Build a summary table of correlations by year (using dni)
summary_data = []

for year in ["All"] + years:
    if year == "All":
        df_subset = big_df
    else:
        df_subset = big_df[big_df["year"] == year]
    
    # Calculate correlations with p-values using dni
    r_dist, p_dist = stats.pearsonr(np.log(df_subset["dist"]), df_subset["dni"])
    r_border, p_border = stats.pearsonr(df_subset["border"], df_subset["dni"])
    r_tariff, p_tariff = stats.pearsonr(np.log(1.0 + 0.01 * df_subset["tariff"]), df_subset["dni"])
    
    summary_data.append({
        "Year": year,
        "N": len(df_subset),
        "Corr(dni, log(dist))": r_dist,
        "p-value (dist)": p_dist,
        "Corr(dni, border)": r_border,
        "p-value (border)": p_border,
        "Corr(dni, log(1+tariff))": r_tariff,
        "p-value (tariff)": p_tariff
    })

summary_df = pd.DataFrame(summary_data)
pd.set_option('display.float_format', '{:.4f}'.format)
summary_df

,Year,N,"Corr(dni, log(dist))",p-value (dist),"Corr(dni, border)",p-value (border),"Corr(dni, log(1+tariff))",p-value (tariff)
0,All,2604,0.4217,0.0000,-0.1832,0.0000,0.3383,0.0000
1,2004,866,0.2784,0.0000,-0.1084,0.0014,0.2606,0.0000
2,2011,868,0.4717,0.0000,-0.1960,0.0000,0.4094,0.0000
3,2017,870,0.5012,0.0000,-0.2395,0.0000,0.4496,0.0000


In [12]:
# Build a summary table of correlations by year using dni2
summary_data_dni2 = []

for year in ["All"] + years:
    if year == "All":
        df_subset = big_df
    else:
        df_subset = big_df[big_df["year"] == year]
    
    # Calculate correlations with p-values using dni2
    r_dist, p_dist = stats.pearsonr(np.log(df_subset["dist"]), df_subset["dni2"])
    r_border, p_border = stats.pearsonr(df_subset["border"], df_subset["dni2"])
    r_tariff, p_tariff = stats.pearsonr(np.log(1.0 + 0.01 * df_subset["tariff"]), df_subset["dni2"])
    
    summary_data_dni2.append({
        "Year": year,
        "N": len(df_subset),
        "Corr(dni2, log(dist))": r_dist,
        "p-value (dist)": p_dist,
        "Corr(dni2, border)": r_border,
        "p-value (border)": p_border,
        "Corr(dni2, log(1+tariff))": r_tariff,
        "p-value (tariff)": p_tariff
    })

summary_df_dni2 = pd.DataFrame(summary_data_dni2)
summary_df_dni2

,Year,N,"Corr(dni2, log(dist))",p-value (dist),"Corr(dni2, border)",p-value (border),"Corr(dni2, log(1+tariff))",p-value (tariff)
0,All,2604,0.5010,0.0000,-0.2084,0.0000,0.3494,0.0000
1,2004,866,0.3714,0.0000,-0.1436,0.0000,0.2462,0.0000
2,2011,868,0.6050,0.0000,-0.2630,0.0000,0.4042,0.0000
3,2017,870,0.5362,0.0000,-0.2221,0.0000,0.4181,0.0000


---
### Regression Analysis

In [13]:
import statsmodels.formula.api as smf

# Regression: dni ~ year fixed effects + border + log(dist) + log(1 + 0.01*tariff)
# Create log-transformed variables
big_df["log_dist"] = np.log(big_df["dist"])
big_df["log_tariff"] = np.log(1.0 + 0.01 * big_df["tariff"])

# Run OLS regression with year fixed effects (using C() for categorical)
model = smf.ols("dni ~ C(year) + border + log_dist + log_tariff", data=big_df)
results = model.fit()

print("Regression: dni ~ fe(year) + border + log(dist) + log(1 + 0.01*tariff)")
print("=" * 70)
print(results.summary())

Regression: dni ~ fe(year) + border + log(dist) + log(1 + 0.01*tariff)
                            OLS Regression Results                            
Dep. Variable:                    dni   R-squared:                       0.212
Model:                            OLS   Adj. R-squared:                  0.211
Method:                 Least Squares   F-statistic:                     139.9
Date:                Mon, 30 Mar 2026   Prob (F-statistic):          1.03e-131
Time:                        10:30:42   Log-Likelihood:                -1114.7
No. Observations:                2604   AIC:                             2241.
Df Residuals:                    2598   BIC:                             2277.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------

In [14]:
# Regression for dni2: dni2 ~ year fixed effects + border + log(dist) + log(1 + 0.01*tariff)

# Run OLS regression with year fixed effects
model_dni2 = smf.ols("dni2 ~ C(year) + border + log_dist + log_tariff", data=big_df)
results_dni2 = model_dni2.fit()

print("Regression: dni2 ~ fe(year) + border + log(dist) + log(1 + 0.01*tariff)")
print("=" * 70)
print(results_dni2.summary())

Regression: dni2 ~ fe(year) + border + log(dist) + log(1 + 0.01*tariff)
                            OLS Regression Results                            
Dep. Variable:                   dni2   R-squared:                       0.287
Model:                            OLS   Adj. R-squared:                  0.286
Method:                 Least Squares   F-statistic:                     209.6
Date:                Mon, 30 Mar 2026   Prob (F-statistic):          3.74e-188
Time:                        10:30:45   Log-Likelihood:                 2149.4
No. Observations:                2604   AIC:                            -4287.
Df Residuals:                    2598   BIC:                            -4252.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------